In [13]:
import torch
import numpy as np
import cv2

def visualize_flattened_tracking(pt_path, save_path='output.mp4', visibility_threshold=0.5, canvas_size=(384, 512), fps=20):
    """
    Visualize flattened pred_tracks and pred_visibility from a .pt file.

    Args:
        pt_path (str): Path to the .pt file
        save_path (str): Output .mp4 file path
        visibility_threshold (float): Only show points with vis > threshold
        canvas_size (tuple): Canvas height and width
        fps (int): Frames per second
    """
    data = torch.load(pt_path, map_location=torch.device('cpu'))
    
    pred_tracks_flat = data['pred_tracks']      # shape [T*N*2]
    pred_visibility_flat = data['pred_visibility']  # shape [T*N]
    T, N = data['shape_info']
    
    # Reshape
    pred_tracks = pred_tracks_flat.reshape(T, N, 2)
    pred_visibility = pred_visibility_flat.reshape(T, N)

    H, W = canvas_size
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(save_path, fourcc, fps, (W, H))

    for t in range(T):
        frame = np.zeros((H, W, 3), dtype=np.uint8)

        for n in range(N):
            vis = pred_visibility[t, n]
            if vis > visibility_threshold:
                x, y = pred_tracks[t, n]
                x, y = int(x), int(y)
                if 0 <= x < W and 0 <= y < H:
                    cv2.circle(frame, (x, y), radius=2, color=(0, 255, 0), thickness=-1)

        out.write(frame)

    out.release()
    print(f"✅ Saved video to {save_path}")

In [15]:
visualize_flattened_tracking(
    pt_path='/Users/mrinalraj/Downloads/WebDownload/Driving48/_8Vy3dlHg2w_00200_tracking.pt',
    save_path='track_video.mp4',
    visibility_threshold=0.0,
    canvas_size=(384, 512),
    fps=20
)

✅ Saved video to track_video.mp4


In [16]:
import torch
import numpy as np
import cv2

def visualize_pred_tracks_only(pt_path, save_path='output.mp4', canvas_size=(384, 512), fps=20, point_color=(0, 255, 0)):
    """
    Visualize tracked motion points from flattened pred_tracks (no visibility used).

    Args:
        pt_path (str): Path to the .pt file
        save_path (str): Output path for video file
        canvas_size (tuple): (height, width) of canvas
        fps (int): Frames per second
        point_color (tuple): BGR color for points (default green)
    """
    data = torch.load(pt_path, map_location=torch.device('cpu'))
    
    pred_tracks_flat = data['pred_tracks']      # shape [T * N * 2]
    T, N = data['shape_info']                   # shape_info = (T, N)
    
    pred_tracks = pred_tracks_flat.reshape(T, N, 2)  # [T, N, 2]

    H, W = canvas_size
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(save_path, fourcc, fps, (W, H))

    for t in range(T):
        frame = np.zeros((H, W, 3), dtype=np.uint8)

        for n in range(N):
            x, y = pred_tracks[t, n]
            x, y = int(x), int(y)
            if 0 <= x < W and 0 <= y < H:
                cv2.circle(frame, (x, y), radius=2, color=point_color, thickness=-1)

        out.write(frame)

    out.release()
    print(f"✅ Video saved to: {save_path}")

In [29]:
visualize_pred_tracks_only(
    pt_path='/Users/mrinalraj/Downloads/WebDownload/Driving48/videosTensors1000/_tigfCJFLZg_00020_tracking.pt',
    save_path='only_tracks_video.mp4',
    canvas_size=(384, 512),
    fps=20
)

✅ Video saved to: only_tracks_video.mp4


In [30]:
data = torch.load('/Users/mrinalraj/Downloads/WebDownload/Driving48/videosTensors1000/_tigfCJFLZg_00020_tracking.pt', map_location=torch.device('cpu'))

pred_tracks_flat = data['pred_tracks']      # shape [T * N * 2]
pred_visibility_flat = data['pred_visibility']  # shape [T*N]
T, N = data['shape_info']                   # shape_info

In [31]:
pred_tracks_flat

tensor([144.0000, 275.0000, 200.0000,  ...,  11.9140, 226.9229, 143.9100])

In [32]:
pred_tracks = pred_tracks_flat.reshape(T, N, 2)
print(pred_tracks)

tensor([[[144.0000, 275.0000],
         [200.0000, 252.0000],
         [184.0000, 276.0000],
         ...,
         [173.0000, 242.0000],
         [202.0000, 273.0000],
         [195.0000, 220.0000]],

        [[146.4170, 274.0607],
         [202.0886, 251.3890],
         [186.2846, 275.4174],
         ...,
         [174.8832, 241.9068],
         [204.4172, 272.7703],
         [192.5711, 219.9550]],

        [[151.5421, 271.8393],
         [207.4720, 248.3223],
         [191.4873, 273.1431],
         ...,
         [180.8713, 239.5087],
         [209.9449, 270.4410],
         [191.2160, 215.7622]],

        ...,

        [[345.6283,   8.7605],
         [433.1870,  12.9184],
         [404.4250,  17.2802],
         ...,
         [392.2374, -18.1863],
         [425.9928,  17.0864],
         [226.8640, 144.2590]],

        [[348.5236,   7.0559],
         [435.6231,  10.3874],
         [405.4482,  16.5254],
         ...,
         [392.2446, -26.2224],
         [427.0008,  12.7695],
         

In [33]:
pred_visibility = pred_visibility_flat.reshape(T, N)
print(pred_visibility)

tensor([[ True,  True,  True,  ...,  True,  True,  True],
        [ True,  True,  True,  ...,  True,  True, False],
        [ True,  True,  True,  ...,  True,  True, False],
        ...,
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False]])


In [34]:
pred_visibility.shape

torch.Size([95, 1000])

In [47]:
pred_visibility[94].any()==True

tensor(False)

In [48]:
for i, vis in enumerate(pred_visibility):
    count = np.count_nonzero(vis > 0)
    print(f"Frame {i}: {count} visible points")

Frame 0: 1000 visible points
Frame 1: 785 visible points
Frame 2: 636 visible points
Frame 3: 643 visible points
Frame 4: 625 visible points
Frame 5: 615 visible points
Frame 6: 594 visible points
Frame 7: 582 visible points
Frame 8: 566 visible points
Frame 9: 544 visible points
Frame 10: 528 visible points
Frame 11: 491 visible points
Frame 12: 529 visible points
Frame 13: 530 visible points
Frame 14: 524 visible points
Frame 15: 498 visible points
Frame 16: 490 visible points
Frame 17: 477 visible points
Frame 18: 468 visible points
Frame 19: 465 visible points
Frame 20: 468 visible points
Frame 21: 465 visible points
Frame 22: 452 visible points
Frame 23: 439 visible points
Frame 24: 447 visible points
Frame 25: 453 visible points
Frame 26: 454 visible points
Frame 27: 448 visible points
Frame 28: 444 visible points
Frame 29: 454 visible points
Frame 30: 446 visible points
Frame 31: 431 visible points
Frame 32: 435 visible points
Frame 33: 425 visible points
Frame 34: 434 visible p

In [53]:
import torch
import numpy as np
import cv2

def visualize_only_visible_points(pt_path, save_path='visible_only.mp4', canvas_size=(384, 512), fps=20, point_color=(0, 255, 0)):
    """
    Visualize only visible tracked points per frame from a .pt file.

    Args:
        pt_path (str): Path to the .pt file
        save_path (str): Output path for video file
        canvas_size (tuple): (height, width)
        fps (int): Frames per second
        point_color (tuple): Color of points in BGR
    """
    data = torch.load(pt_path, map_location=torch.device('cpu'))
    
    pred_tracks_flat = data['pred_tracks']          # [T * N * 2]
    pred_visibility_flat = data['pred_visibility']  # [T * N]
    T, N = data['shape_info']

    # Reshape
    pred_tracks = pred_tracks_flat.reshape(T, N, 2)
    pred_visibility = pred_visibility_flat.reshape(T, N)

    H, W = canvas_size
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(save_path, fourcc, fps, (W, H))

    for t in range(T):
        frame = np.zeros((H, W, 3), dtype=np.uint8)
        
        visible_indices = pred_visibility[t] == False
        visible_points = pred_tracks[t][visible_indices]  # shape: [M, 2]

        for x, y in visible_points:
            x, y = int(x), int(y)
            if 0 <= x < W and 0 <= y < H:
                cv2.circle(frame, (x, y), radius=2, color=point_color, thickness=-1)

        out.write(frame)

    out.release()
    print(f"✅ Video saved showing only visible points: {save_path}")

In [54]:
visualize_only_visible_points(
    pt_path='/Users/mrinalraj/Downloads/WebDownload/Driving48/videosTensors1000/_tigfCJFLZg_00020_tracking.pt',
    save_path='visible_points_video.mp4',
    canvas_size=(384, 512),
    fps=20
)

✅ Video saved showing only visible points: visible_points_video.mp4
